## 概要

実習3_4です。与えられたデータを基に学習を深めてみましょう。

## 業務シナリオの紹介

あなたはある医療機関に勤務しており、乳癌（にゅうがん、英: Breast cancer）の検出を改善したいと考えています。

機械学習 (ML) を利用してこの問題を解決するのがあなたの課題です。このデータセットを使用して ML モデルのトレーニングを行い、患者に異常があるかどうかを予測します。

## このデータセットについて

breast_cancer.csv　は、Pythonのscikit-learn付属データセットに用意されており、ダウンロードしたデータです。

## 属性情報

PowerPointをご覧ください。
このデータセットの詳細については、 (https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) を参照してください。

## データのインポート

次のセルを実行するとデータがインポートされ、使用できる状態になります。

**注:** 次のセルは、以前のラボでの重要なステップです。

In [3]:
# 警告メッセージを無視するためにwarningsモジュールを使用
import warnings, requests, zipfile, io

# 警告メッセージを無視するフィルターを設定
warnings.simplefilter('ignore')

# データ処理のためにpandasモジュールをインポート
import pandas as pd

In [4]:
# boto3を使ってAWSのサービスにアクセスするためのライブラリをインポート
import boto3

# pandasライブラリを使って、AWS S3上のCSVファイルを読み込む
# 事前に必要なデータをS3にアップロードをして、URIをメモしておいてください。
df = pd.read_csv('s3://machinelearning0105/breast_cancer.csv')

# ステップ 1: データの調査
まず、データセット内のデータを簡単に再確認します。

このラボを最大限活用するために、セルを実行する前に、手順とコードを注意深くお読みください。慌てずに実験を進めましょう。

まず、**shape** を使って、行と列の数を調べます。

In [5]:
# DataFrameの行数と列数を確認するためにshape属性を使用
df.shape

(569, 31)

次に、列のリストを見てみましょう。

In [6]:
# DataFrameの列名を表示するためにcolumns属性を使用
df.columns

Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension',
       'target'],
      dtype='object')

30の特徴量と、ターゲット列の名前が *target* であることがわかります。


# ステップ 2: データの準備

このラボのために、データを 3 つのデータセットに分割する必要があります。

インターネット検索で、データセットを分割するさまざまな方法を調べることができます。最も多く見つかるコードサンプルは、データセットを *ターゲット* と *特徴量* に分割するものです。次に、これら 2 つのデータセットを 3 つのサブセットに分割し、合計 6 つのデータセットを使って追跡します。

## ターゲット列の位置の移動

XGBoost では、トレーニングデータは 1 つのファイルに格納されている必要があります。ファイルの中で、ターゲット値は必ず最初の列にします。

ターゲット列を選択して先頭に移動します。

In [7]:
# DataFrameの列名をリストに変換
cols = df.columns.tolist()

# 先頭の列を末尾に移動するためにスライスを使用
cols = cols[-1:] + cols[:-1]

# 列の順序を変更するためにDataFrameを再構築
df = df[cols]

**target** が先頭の列になっていれば正しい状態です。

In [8]:
# DataFrameの列名を表示するためにcolumns属性を使用
df.columns

Index(['target', 'mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension'],
      dtype='object')

## データの分割

まず、データセットを 2 つのデータセットに分割します。1 つのデータセットをトレーニングに使用し、もう 1 つのデータセットは再度分割して検証とテストに使用します。

*train_test_split 関数* は、Python 用の無料機械学習ライブラリである *scikit-learn ライブラリ* のものを使用します。ここには皆さんが使用するものを含めて、多くのアルゴリズムと便利な関数が揃っています。

- 関数の詳細については、[Train_Test_Split ドキュメント] (https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) を参照してください。
 - scikit-learn の詳細については、[scikit-learn ガイド] (https://scikit-learn.org/stable/) を参照してください。

データ量が多くないので、分割したデータセットに各クラスを代表する量のデータが含まれていることの確認もできます。そこで *階層化* スイッチを使用します。最後に、繰り返し分割できるように乱数を使用します。

In [9]:
# 機械学習モデルのトレーニングとテストのためにデータを分割するためにtrain_test_split関数をインポート
from sklearn.model_selection import train_test_split

# データセットをトレーニングセットとテスト＆検証セットに分割するためにtrain_test_split関数を使用
# test_sizeでテスト＆検証セットの割合を指定し、random_stateで乱数のシードを固定
# stratifyでクラスの分布を保持して分割
train, test_and_validate = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

次に、*test_and_validate* データセットを 2 つに等分します。

In [10]:
# train_test_split関数を使用してデータを分割
# test_and_validateデータをtestとvalidateに分割し、test_sizeで分割比率を指定
# random_stateで再現性を確保し、stratifyで層別サンプリングを行う（'class'列を考慮）
test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['target'])

3 つのデータセットを調べます。

In [11]:
# 訓練データセットの行数と列数を表示
print(train.shape)

# テストデータセットの行数と列数を表示
print(test.shape)

# 検証データセットの行数と列数を表示
print(validate.shape)

(455, 31)
(57, 31)
(57, 31)


ここでクラスの分布を確認します。

In [12]:
# 'target'列の値の出現回数を表示するためにvalue_countsメソッドを使用
print(train['target'].value_counts())

# テストデータの'target'列の値の出現回数を表示
print(test['target'].value_counts())

# 検証データの'target'列の値の出現回数を表示
print(validate['target'].value_counts())

target
1    285
0    170
Name: count, dtype: int64
target
1    36
0    21
Name: count, dtype: int64
target
1    36
0    21
Name: count, dtype: int64


## Amazon S3 にデータをアップロード

XGboost では、Amazon Simple Storage Service (Amazon S3) からトレーニング用のデータをロードします。したがって、データをカンマ区切り値 (CSV) ファイルに書き込み、そのファイルを Amazon S3 にアップロードする必要があります。

まず、S3 バケットにいくつかの変数を設定してから、CSV ファイルを Amazon S3 にアップロードする関数を作成します。この関数は再利用できます。

まず、関数を調べます。

次のラインに注目してください。

`dataframe.to_csv(csv_buffer, header=False, index=False)`

このラインは、関数に渡された pandas DataFrame を *csv_buffer* という名前の IO バッファに書き込みます。ファイルをローカルに書き込む必要がないため、バッファを使用します。

列ヘッダーが書き出されないようにするには、`header=False` を使用します。pandas index が出力されないようにするには、`index=False` を使用します。

csv_buffer をオブジェクトとして Amazon S3 に書き込むには、`object` で `put` オペレーションを使用します。これは `bucket` のプロパティです。


In [15]:
# S3バケットとプレフィックスの設定
bucket='machinelearning0105'
prefix='lab3'

# 訓練データ、テストデータ、検証データのファイル名設定
train_file='vertebral_train.csv'
test_file='vertebral_test.csv'
validate_file='vertebral_validate.csv'

# osモジュールを使用してファイルパスを構築
import os

# boto3セッションからS3リソースを取得
s3_resource = boto3.Session().resource('s3')

# S3にCSVファイルをアップロードするための関数を定義
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

作成した関数を使用して、3 つのデータセットをアップロードします。

In [16]:
# train_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(train_file, 'train', train)

# test_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(test_file, 'test', test)

# validate_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(validate_file, 'validate', validate)

# ステップ 3: モデルのトレーニング

Amazon S3 にデータをアップロードできたので、モデルをトレーニングしましょう。

最初のステップでは、XGBoost コンテナの URI を取得します。

In [17]:
# Amazon SageMaker用のモジュールをインポート
import boto3
from sagemaker.image_uris import retrieve

# SageMakerにおいてXGBoostアルゴリズムを利用するためのコンテナイメージを取得
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


次に、モデルにいくつかの *ハイパーパラメータ* を設定する必要があります。モデルのトレーニングは初めてなので、まずはいくつかの値でやってみましょう。

In [18]:
# XGBoostのハイパーパラメータを設定するための辞書を作成
hyperparams={"num_round":"42",
             "eval_metric": "auc",
             "objective": "binary:logistic"}

**estimator** 関数を使用してモデルを設定します。重要なパラメータをいくつか挙げます。

- **instance_count** - トレーニングに使用するインスタンスの数を定義します。ここで使用するインスタンスは *1 つ* です。
- **instance_type** - トレーニングのインスタンスタイプを定義します。このケースでは、*ml.m4.xlarge* です。


In [19]:
# SageMaker SDKを使用して、XGBoostのモデルをトレーニングおよびデプロイするために必要なモジュールをインポート
import sagemaker

# 出力先のS3ロケーションを設定
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

# SageMakerのXGBoostアルゴリズムを使用するためにXGBoost Estimatorを作成
# container: 使用するコンテナの指定
# sagemaker.get_execution_role(): SageMakerの実行ロールを取得
# instance_count: モデルトレーニングに使用するインスタンスの数を指定
# instance_type: モデルトレーニングに使用するインスタンスのタイプを指定
# output_path: トレーニングアーティファクト（モデルアーティファクト）のS3出力先を指定
# hyperparameters: XGBoostのハイパーパラメータを指定
# sagemaker_session: SageMakerセッションを指定
xgb_model=sagemaker.estimator.Estimator(container,
                                       sagemaker.get_execution_role(),
                                       instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       output_path=s3_output_location,
                                        hyperparameters=hyperparams,
                                        sagemaker_session=sagemaker.Session())

推定器には、モデルにデータを供給するための *チャネル* が必要です。トレーニングでは、*train_channel* と *validate_channel* を使用します。

In [20]:
# SageMakerのトレーニングジョブに使用するトレーニングデータのS3パスとコンテントタイプを指定
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

# SageMakerのトレーニングジョブに使用する検証データのS3パスとコンテントタイプを指定
validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

# トレーニングジョブのデータチャネルにトレーニングデータと検証データを指定
data_channels = {'train': train_channel, 'validation': validate_channel}

**fit** を実行するとモデルがトレーニングされます。

**注:** この処理は最長 5 分かかることがあります。

In [21]:
# XGBoostモデルを学習させるためにfitメソッドを使用
# inputsにはデータチャネルを指定し、logsをFalseに設定してログを表示しないようにする
xgb_model.fit(inputs=data_channels, logs=False)

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2024-01-05-07-10-19-612



2024-01-05 07:10:20 Starting - Starting the training job....
2024-01-05 07:10:47 Starting - Preparing the instances for training....................
2024-01-05 07:12:31 Downloading - Downloading input data.....
2024-01-05 07:13:01 Downloading - Downloading the training image.........
2024-01-05 07:13:51 Training - Training image download completed. Training in progress.....
2024-01-05 07:14:17 Uploading - Uploading generated training model.
2024-01-05 07:14:28 Completed - Training job completed
